<a href="https://colab.research.google.com/github/springboardmentor1000-del/RideWise-BikeDemandPrediction2/blob/Kavyasri/Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install xgboost lightgbm


In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor


In [3]:
day = pd.read_csv("/content/day[1].csv")

# ---- Feature Engineering ----
day["is_weekend"] = day["weekday"].apply(lambda x: 1 if x in [0, 6] else 0)
day["comfort_index"] = day["temp"] * (1 - day["hum"])
day["wind_chill_effect"] = day["windspeed"] * (1 - day["temp"])

weather_map = {1: 0, 2: 1, 3: 2, 4: 3}
day["weather_severity"] = day["weathersit"].map(weather_map)


In [4]:
day_features = [
    "season", "yr", "mnth", "workingday",
    "is_weekend", "temp", "comfort_index",
    "wind_chill_effect", "weather_severity"
]

X_day = day[day_features]
y_day = day["cnt"]


In [5]:
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_day, y_day, test_size=0.2, random_state=42
)


In [6]:
xgb_day = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

xgb_day.fit(X_train_d, y_train_d)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=400,
             n_jobs=None, num_parallel_tree=None, ...)

In [7]:
day_preds = xgb_day.predict(X_test_d)

day_rmse = np.sqrt(mean_squared_error(y_test_d, day_preds))
day_r2 = r2_score(y_test_d, day_preds)

print("DAY MODEL (XGBoost)")
print("RMSE:", day_rmse)
print("R2 Score:", day_r2)


DAY MODEL (XGBoost)
RMSE: 623.067838401245
R2 Score: 0.9031857848167419


In [8]:
joblib.dump(xgb_day, "ridewise_day_xgboost.pkl")


['ridewise_day_xgboost.pkl']

In [9]:
hour = pd.read_csv("/content/hour[1].csv")

# ---- Feature Engineering ----
hour["is_weekend"] = hour["weekday"].apply(lambda x: 1 if x in [0, 6] else 0)
hour["is_peak_hour"] = hour["hr"].apply(
    lambda x: 1 if (7 <= x <= 9 or 17 <= x <= 19) else 0
)

hour["comfort_index"] = hour["temp"] * (1 - hour["hum"])
hour["wind_chill_effect"] = hour["windspeed"] * (1 - hour["temp"])
hour["weather_severity"] = hour["weathersit"].map(weather_map)


In [10]:
hour_features = [
    "hr", "is_peak_hour", "season", "workingday",
    "is_weekend", "temp", "comfort_index",
    "wind_chill_effect", "weather_severity",
    "hum", "windspeed"
]

X_hour = hour[hour_features]
y_hour = hour["cnt"]


In [11]:
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_hour, y_hour, test_size=0.2, random_state=42
)


In [12]:
lgbm_hour = LGBMRegressor(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgbm_hour.fit(X_train_h, y_train_h)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013804 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 709
[LightGBM] [Info] Number of data points in the train set: 13903, number of used features: 11
[LightGBM] [Info] Start training from score 190.577070


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, n_estimators=400,
              random_state=42, subsample=0.8)

In [13]:
hour_preds = lgbm_hour.predict(X_test_h)

hour_rmse = np.sqrt(mean_squared_error(y_test_h, hour_preds))
hour_r2 = r2_score(y_test_h, hour_preds)

print("HOUR MODEL (LightGBM)")
print("RMSE:", hour_rmse)
print("R2 Score:", hour_r2)


HOUR MODEL (LightGBM)
RMSE: 67.52374946838881
R2 Score: 0.8560114475121191


In [14]:
joblib.dump(lgbm_hour, "ridewise_hour_lightgbm.pkl")


['ridewise_hour_lightgbm.pkl']